# BƯỚC 7 — Phân lớp dự đoán tái nhập viện dưới 30 ngày

**Đề tài 5 — Diabetes 130-US Hospitals (1999–2008)**. Sử dụng đúng ma trận sparse, nhãn và danh sách bệnh nhân đã chia ở Bước 4. **Không đụng đến tập test trong bước này**: chỉ dùng *train* để học và *validation* để so sánh mô hình, chọn ngưỡng. Đánh giá cuối cùng được dành cho bước đánh giá cuối.

**Câu hỏi nghiệp vụ:** Sau xuất viện, nên ưu tiên gọi theo dõi bệnh nhân nào? Đây là thử nghiệm nghiên cứu với dữ liệu lịch sử; không sử dụng kết quả để quyết định chăm sóc thực tế.

## 1. Thư viện và đường dẫn
Chạy JupyterLab tại `D:\Readmission_Project`, Notebook đặt ở thư mục `notebooks`. Đường dẫn được xác định tương đối theo thư mục hiện hành hoặc thư mục gốc.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from scipy import sparse
import matplotlib.pyplot as plt
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (precision_score, recall_score, f1_score, balanced_accuracy_score,
                             average_precision_score, roc_auc_score, confusion_matrix,
                             precision_recall_curve, ConfusionMatrixDisplay)
import joblib

RANDOM_STATE = 42
cwd = Path.cwd()
ROOT = next((p for p in (cwd, cwd.parent, Path(r'D:\Readmission_Project'))
             if (p/'data'/'processed'/'step4'/'X_train.npz').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Không tìm thấy X_train.npz. Kiểm tra đã hoàn thành Bước 4 và cấu trúc thư mục.')
DATA = ROOT/'data'/'processed'/'step4'
REPORT = ROOT/'reports'/'classification'
REPORT.mkdir(parents=True, exist_ok=True)
print('ROOT:', ROOT)
print('DATA:', DATA)
print('REPORT:', REPORT)

## 2. Nạp dữ liệu và kiểm tra toàn vẹn
Giữ `test` chưa sử dụng để đánh giá mô hình; chỉ kiểm tra rằng file tồn tại. Không đọc nhãn của tập test.

In [ ]:
X_train = sparse.load_npz(DATA/'X_train.npz').tocsr()
X_val = sparse.load_npz(DATA/'X_validation.npz').tocsr()
y_train = np.load(DATA/'y_train.npy')
y_val = np.load(DATA/'y_validation.npy')
train_ids = pd.read_csv(DATA/'train_ids_labels.csv')
val_ids = pd.read_csv(DATA/'validation_ids_labels.csv')
for name in ['X_test.npz','y_test.npy','test_ids_labels.csv','feature_names.csv']:
    assert (DATA/name).exists(), f'Thiếu file: {name}'
assert X_train.shape[0] == len(y_train) == len(train_ids)
assert X_val.shape[0] == len(y_val) == len(val_ids)
assert X_train.shape[1] == X_val.shape[1]
assert set(np.unique(y_train)).issubset({0,1}) and set(np.unique(y_val)).issubset({0,1})
assert set(train_ids.patient_nbr).isdisjoint(set(val_ids.patient_nbr)), 'Lỗi: trùng bệnh nhân giữa train và validation'
assert np.array_equal(train_ids.readmitted_30d.to_numpy(), y_train)
assert np.array_equal(val_ids.readmitted_30d.to_numpy(), y_val)
feature_names = pd.read_csv(DATA/'feature_names.csv')['feature'].astype(str).to_numpy()
assert len(feature_names)==X_train.shape[1]
print('Train:', X_train.shape, '| Positive:', int(y_train.sum()), '| Tỷ lệ:', round(y_train.mean()*100,2),'%')
print('Validation:', X_val.shape, '| Positive:', int(y_val.sum()), '| Tỷ lệ:', round(y_val.mean()*100,2),'%')
print('Số đặc trưng:', len(feature_names))
print('Train/validation không trùng bệnh nhân; test chưa đọc.')

## 3. Baseline và các mô hình phân lớp
- **Baseline:** luôn dự đoán lớp đa số. Đánh giá cả đường điểm xác suất của baseline.
- **Decision Tree:** dễ diễn giải, dùng độ sâu hạn chế để hạn chế quá khớp.
- **Random Forest:** đối chứng hiệu năng; giới hạn độ sâu và số cây để phù hợp laptop.

Không sử dụng `accuracy` làm chỉ số chính vì tỷ lệ nhãn dương thấp.

In [ ]:
models = {
    'Baseline_majority': DummyClassifier(strategy='most_frequent'),
    'DecisionTree': DecisionTreeClassifier(max_depth=7, min_samples_leaf=100,
                                           class_weight='balanced', random_state=RANDOM_STATE),
    'RandomForest': RandomForestClassifier(n_estimators=100, max_depth=12,
                                           min_samples_leaf=40, class_weight='balanced_subsample',
                                           random_state=RANDOM_STATE, n_jobs=-1),
}
for name, model in models.items():
    print('Đang huấn luyện:', name)
    model.fit(X_train, y_train)
print('Huấn luyện xong.')

## 4. So sánh các chỉ số trên validation
Dùng ngưỡng mặc định 0,50 để so sánh ban đầu. **Average Precision (AP)** là diện tích dạng tổng có trọng số theo recall của đường precision–recall (không đồng nhất hoàn toàn với cách tính PR-AUC hình thang). Luôn đối chiếu tỷ lệ nhãn dương và baseline.

In [ ]:
scores = {}
rows=[]
for name, model in models.items():
    prob=model.predict_proba(X_val)[:,list(model.classes_).index(1)]
    pred=(prob>=0.5).astype(int)
    scores[name]=prob
    rows.append({'model':name,'threshold':0.5,
        'precision':precision_score(y_val,pred,zero_division=0),
        'recall':recall_score(y_val,pred,zero_division=0),
        'f1':f1_score(y_val,pred,zero_division=0),
        'balanced_accuracy':balanced_accuracy_score(y_val,pred),
        'average_precision':average_precision_score(y_val,prob),
        'roc_auc':roc_auc_score(y_val,prob),
        'TP':int(((pred==1)&(y_val==1)).sum()), 'FP':int(((pred==1)&(y_val==0)).sum()),
        'TN':int(((pred==0)&(y_val==0)).sum()), 'FN':int(((pred==0)&(y_val==1)).sum())})
metrics=pd.DataFrame(rows).sort_values('average_precision',ascending=False)
display(metrics.round(4))
metrics.to_csv(REPORT/'validation_metrics_threshold_050.csv',index=False,encoding='utf-8-sig')
print('AP tham chiếu mô hình ngẫu nhiên gần tỷ lệ nhãn dương:',round(y_val.mean(),4))

## 5. Đường Precision–Recall và ma trận nhầm lẫn

In [ ]:
fig,ax=plt.subplots(figsize=(8,5))
for name,prob in scores.items():
    p,r,_=precision_recall_curve(y_val,prob)
    ax.plot(r,p,label=f'{name} (AP={average_precision_score(y_val,prob):.3f})')
ax.axhline(y_val.mean(),linestyle='--',label=f'Tỷ lệ nhãn dương ({y_val.mean():.3f})')
ax.set(xlabel='Recall',ylabel='Precision',title='Precision–Recall trên validation')
ax.legend();fig.tight_layout()
fig.savefig(REPORT/'01_precision_recall_validation.png',dpi=160)
plt.show()
fig,axs=plt.subplots(1,len(models),figsize=(13,3.8))
for ax,(name,prob) in zip(axs,scores.items()):
    ConfusionMatrixDisplay(confusion_matrix(y_val,prob>=0.5),display_labels=['Không <30','<30']).plot(ax=ax,colorbar=False)
    ax.set_title(name)
fig.tight_layout();fig.savefig(REPORT/'02_confusion_matrices_050.png',dpi=160)
plt.show()

## 6. Chọn ngưỡng dựa trên validation theo năng lực gọi theo dõi
Một cách lựa chọn thực tế: bệnh viện chỉ có thể gọi **10% lượt xuất viện**. Xếp hạng theo xác suất dự đoán và đo Precision@10%, Recall@10%, Lift@10% trên validation. **Không học ngưỡng bằng tập test.** Nếu hai mô hình gần nhau, cân nhắc ưu tiên cây quyết định dễ giải thích.

In [ ]:
top_fraction=0.10
k=max(1,int(np.ceil(len(y_val)*top_fraction)))
capacity=[]
for name,prob in scores.items():
    selected=np.argsort(-prob,kind='stable')[:k]
    hits=int(y_val[selected].sum())
    capacity.append({'model':name,'top_fraction':top_fraction,'capacity_calls':k,
                     'true_readmissions_captured':hits,
                     'precision_at_10pct':hits/k,
                     'recall_at_10pct':hits/int(y_val.sum()),
                     'lift_at_10pct':(hits/k)/y_val.mean()})
capdf=pd.DataFrame(capacity).sort_values(['recall_at_10pct','precision_at_10pct'],ascending=False)
display(capdf.round(4))
capdf.to_csv(REPORT/'validation_capacity_top10pct.csv',index=False,encoding='utf-8-sig')
chosen_name=capdf.iloc[0]['model']
print('Ứng viên theo Recall@10% validation:',chosen_name)
print('Lưu ý: nếu lift gần 1 hoặc thấp hơn baseline, cần diễn giải hạn chế của mô hình.')

## 7. Giải thích đặc trưng và lưu mô hình
Feature importance đo đóng góp trong quá trình tách nhánh của mô hình cây, **không suy ra quan hệ nhân quả**. Với các biến danh mục One-Hot, mỗi cột biểu diễn một mục đã mã hóa.

In [ ]:
for name in ['DecisionTree','RandomForest']:
    model=models[name]
    importance=pd.DataFrame({'feature':feature_names,'importance':model.feature_importances_})
    importance=importance.sort_values('importance',ascending=False)
    importance.to_csv(REPORT/f'feature_importance_{name}.csv',index=False,encoding='utf-8-sig')
    print('\nTop 12:',name)
    display(importance.head(12))
for name,model in models.items():
    joblib.dump(model,REPORT/f'model_{name}.joblib')
print('Đã lưu 3 mô hình và các bảng feature importance.')

## 8. Nhật ký quyết định và kiểm tra kết quả
**Lưu ý đạo đức:** mô hình huấn luyện trên dữ liệu bệnh viện lịch sử của Mỹ, không dùng để phân bổ chăm sóc thật. Bước tiếp theo cần đánh giá recall theo giới tính/chủng tộc; xem xét missing/Unknown và hậu quả bỏ sót bệnh nhân. Tập test giữ nguyên cho đánh giá cuối sau khi cố định mô hình và chính sách ngưỡng.

In [ ]:
decision_log=pd.DataFrame([
['D07-01','Dùng ma trận Bước 4; fit train; đánh giá validation','Không tái fit trên validation/test; tránh leakage','Độ tin cậy đánh giá'],
['D07-02','Baseline dự đoán lớp đa số','Bắt buộc so sánh với phương án đơn giản','Recall, precision, AP'],
['D07-03','Cây quyết định vs Random Forest; random_state=42','Có mô hình giải thích được và mô hình đối chứng','AP, recall, F1'],
['D07-04','Giữ ngưỡng 0,50 cho bảng đầu; so top 10% theo năng lực gọi','Không mặc định ngưỡng 0,5 phù hợp thực tế','Precision@10%, Recall@10%'],
['D07-05','Không đọc test để tối ưu','Giữ một tập đánh giá cuối độc lập','Giảm lạc quan do lựa chọn mô hình'],
['D07-06','Importance chỉ là liên hệ dự đoán','Không khẳng định thuốc/giới tính gây tái nhập viện','Diễn giải/đạo đức']
],columns=['ma','quyet_dinh','ly_do','anh_huong'])
decision_log.to_csv(REPORT/'decision_log_step7.csv',index=False,encoding='utf-8-sig')
summary={'n_train':int(len(y_train)),'n_validation':int(len(y_val)),
         'n_features':int(X_train.shape[1]),'validation_prevalence':float(y_val.mean()),
         'top_fraction':top_fraction,'selected_on_validation':chosen_name,
         'test_used':False,'random_state':RANDOM_STATE}
(REPORT/'training_summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8')
assert all((REPORT/f'model_{name}.joblib').exists() for name in models)
assert (REPORT/'validation_metrics_threshold_050.csv').exists()
print('Đã lưu kết quả:',REPORT)
print('Train:',X_train.shape,'| validation:',X_val.shape)
print('HOÀN THÀNH BƯỚC 7 (huấn luyện và validation; test chưa sử dụng).')

### Diễn giải kết quả
- Nếu **Average Precision** chỉ gần tỷ lệ dương (~11%), mô hình chưa đủ hữu ích để phân loại ưu tiên.
- So sánh Recall@10% với khả năng gọi 10% bệnh nhân, luôn báo kèm số bệnh nhân được gọi.
- **Không sử dụng mô hình cho quyết định chăm sóc thật** khi chưa kiểm định ngoài mẫu, fairness và giám sát chuyên môn.
- Gửi các bảng `validation_metrics_threshold_050.csv` và `validation_capacity_top10pct.csv` để phân tích trước bước đánh giá cuối.